In [1]:
import os
import gc
import glob
import random
import warnings
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms

from PIL import Image
from PIL import Image as PILImage
from tqdm.auto import tqdm

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import preprocess_image

from lime import lime_image
import shap

import quantus
from quantus import MaxSensitivity, norm_func, similarity_func
import matplotlib.pyplot as plt

from scipy.stats import spearmanr


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Fixed 224x224 inputs -> let cuDNN pick the fastest conv algorithms
torch.backends.cudnn.benchmark = True

Using device: cuda


In [4]:
name = "aveera"

In [5]:
IDRID_DIR = f"/data/{name}/Refined IDRiD"
IDRID_SPLIT_DIRS = [
    os.path.join(IDRID_DIR, "Test", "Test"),
    os.path.join(IDRID_DIR, "Train", "Train"),
]

drive_model_dir = f"/data/{name}/models"
RESULTS_DIR = f"/data/{name}/results"
CHECKPOINT_DIR = f"/data/{name}/results/maxsens_checkpoints"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for split_dir in IDRID_SPLIT_DIRS:
    assert os.path.isdir(os.path.join(split_dir, "Images")), (
        f"Expected an 'Images' folder inside {split_dir} -- check IDRID_DIR."
    )
print(f"Using Refined IDRiD data from '{IDRID_DIR}' (Test + Train splits)")


Using Refined IDRiD data from '/data/aveera/Refined IDRiD' (Test + Train splits)


**Memory hygiene helpers**

In [6]:
import ctypes
import resource

try:
    _libc = ctypes.CDLL("libc.so.6")
except OSError:
    _libc = None


def clear_memory():
    """Reclaim Python and CUDA memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    if _libc is not None:
        _libc.malloc_trim(0)


def log_rss(tag=""):
    """Log resident set size (the number SLURM's OOM killer actually watches)."""
    rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  
    msg = f"[rss{' ' + tag if tag else ''}] peak RSS so far = {rss_mb:.0f} MB"
    if torch.cuda.is_available():
        msg += f"  |  GPU allocated = {torch.cuda.memory_allocated()/1e6:.0f} MB"
    print(msg)


def unload_model(model=None, engine=None, engine_cache=None):
    """Fully detach a model + its AttributionEngine from memory before loading the next one."""
    if engine is not None and engine_cache is not None:
        engine_cache.pop(id(model), None)
        engine._gradcam = None
        engine.shap_explainer = None
        del engine
    if model is not None:
        model.to("cpu")
        del model
    clear_memory()


def gpu_mem_report(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"[mem{' ' + tag if tag else ''}] allocated={alloc:.2f} GB  reserved={reserved:.2f} GB")


**Loading and pre-processing Refined IDRiD**

In [7]:
# Pixel values of all lesion classes (background + anatomical structures excluded)
LESION_LABEL_IDS = {
    4,    # Vitreous Hemorrhage (VH)
    63,   # Hard Exudate (EX)
    96,   # IRMA
    127,  # Hemorrhage (HE)
    166,  # Neovascularization (NV)
    191,  # Cotton-Wool Spot (CWS)
    255,  # Microaneurysm (MA)
}


In [8]:
def resize_mask_no_loss(mask, target_size):
    mask_f = mask.astype(np.float32)
    coverage = cv2.resize(mask_f, target_size, interpolation=cv2.INTER_AREA)
    # any nonzero coverage in the box -> lesion touched that region
    return (coverage > 0).astype(np.uint8)

def load_and_preprocess(split_dirs, target_size=(224, 224), max_images=None):

    samples = []
    for split_dir in split_dirs:
        images_dir = os.path.join(split_dir, "Images")
        labels_dir = os.path.join(split_dir, "Labels")

        image_paths = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
        for img_path in image_paths:
            if split_dir == os.path.normpath(f"/data/{name}/Refined IDRiD/Train/Train"): 
              image_id = os.path.splitext(os.path.basename(img_path))[0]+"_vessel"
            else:
              image_id = os.path.splitext(os.path.basename(img_path))[0] 

            mask_path = os.path.join(labels_dir, f"{image_id}.png")
            if not os.path.isfile(mask_path):
                continue

            raw_mask = np.array(Image.open(mask_path).convert('L'))
            raw_img = np.array(Image.open(img_path).convert('RGB'))

            # Resizes the lesion mask if the height and width aren't the same as the original image
            if raw_mask.shape[:2] != raw_img.shape[:2]:
                raw_mask = cv2.resize(raw_mask, (raw_img.shape[1], raw_img.shape[0]),interpolation=cv2.INTER_NEAREST)

            green_channel = raw_img[:, :, 1] 

            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            enhanced_green = clahe.apply(green_channel) 

            bgr_enhanced = cv2.merge([enhanced_green, enhanced_green, enhanced_green]) # Merge into a clean 3-channel layout 
            
            rgb_enhanced = cv2.cvtColor(bgr_enhanced, cv2.COLOR_BGR2RGB)

            img_resized = cv2.resize(rgb_enhanced, (224, 224), interpolation=cv2.INTER_CUBIC)# Resize the image

            # Keep only lesion pixels and exclude background and the 4 anatomical classes
            lesion_mask = np.isin(raw_mask, list(LESION_LABEL_IDS)).astype(np.uint8)

            mask_resized = resize_mask_no_loss(lesion_mask, target_size)

            samples.append((img_resized, mask_resized))

            if max_images is not None and len(samples) >= max_images:
                return samples
        print("There are ", len(samples)," images in ", split_dir)

    return samples

In [9]:
# MaxSensitivity only needs the first N_TEST samples (same value as the original notebook),
# so we cap the load here instead of loading every split image into RAM.
N_TEST = 81

idrid_subset = load_and_preprocess(IDRID_SPLIT_DIRS, max_images=N_TEST)
print(f"Loaded {len(idrid_subset)} IDRiD images (capped at N_TEST={N_TEST})")

maxsens_samples = idrid_subset[:N_TEST]


There are  27  images in  /data/aveera/Refined IDRiD/Test/Test


Loaded 81 IDRiD images (capped at N_TEST=81)


**Model loading**

In [10]:
def load_model(model_name, weight_path, device):
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        num_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_features, 5))
    elif model_name == "efficientnetb0":
        model = models.efficientnet_b0(weights=None)
        num_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_features, 5))
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    model.load_state_dict(torch.load(weight_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model


**Attribution engine**

In [11]:
class AttributionEngine:
    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    CLASS_NAMES = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]

    def __init__(self, model, device, lime_samples=1000, img_size=224, shap_max_evals=500,
                 shap_batch_size=50, predict_chunk_size=32):
        self.model = model.to(device)
        self.model.eval()
        self.device = device

        self.lime_samples    = lime_samples
        self.img_size        = img_size
        self.shap_max_evals  = shap_max_evals
        self.shap_batch_size = shap_batch_size
        self.class_names     = self.CLASS_NAMES
        self.predict_chunk_size = predict_chunk_size

        self._gradcam = None

        self._norm_transform = transforms.Normalize(mean=self.IMAGENET_MEAN.tolist(), std=self.IMAGENET_STD.tolist())
        self.shap_explainer = None
        self.lime_explainer = lime_image.LimeImageExplainer(random_state=np.random.RandomState(SEED))

    def find_gradcam_target_layer(self):
        if hasattr(self.model, "layer4"):
            target_layer = self.model.layer4[-1]
        elif hasattr(self.model, "features"):
            target_layer = self.model.features[-1]
        else:
            conv_layers = [m for m in self.model.modules() if isinstance(m, nn.Conv2d)]
            if not conv_layers:
                raise ValueError("Could not locate a Conv2d layer in the model for Grad-CAM.")
            target_layer = conv_layers[-1]
        return target_layer

    # Builds a Grad-CAM object
    def get_gradcam(self):
        if self._gradcam is None:
            target_layer = self.find_gradcam_target_layer()
            self._gradcam = GradCAM(model=self.model, target_layers=[target_layer])
        return self._gradcam


    def unnormalize_to_uint8(self, img_tensor):
        img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
        img = img * self.IMAGENET_STD + self.IMAGENET_MEAN
        img = np.clip(img, 0.0, 1.0)
        return (img * 255).astype(np.uint8)

    def uint8_to_model_input(self, raw_uint8):
        """Opposite of unnormalize_to_uint8: uint8 H,W,C becomes a normalized (1,3,H,W) tensor on device."""

        t=torch.from_numpy(raw_uint8).float().div(255.0).permute(2,0,1)
        return self._norm_transform(t).unsqueeze(0).to(self.device)

    @staticmethod
    def normalize_map(m):
        m = np.nan_to_num(m.astype(np.float32))
        mn, mx = m.min(), m.max()
        if mx - mn > 1e-12:
            m = (m - mn) / (mx - mn)
        else:
            m = np.zeros_like(m)
        return m


    def resize_map(self, m):
        if m.shape[0] != self.img_size or m.shape[1] != self.img_size:
            m = cv2.resize(m.astype(np.float32), (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        return m

    def compute_gradcam(self, img, target_class):
        cam = self.get_gradcam()
        targets = [ClassifierOutputTarget(target_class)]
        grayscale_cam = cam(input_tensor=img, targets=targets)[0]
        heatmap = self.normalize_map(grayscale_cam)
        return self.resize_map(heatmap)

    def _predict_in_chunks(self, numpy_images):
        """Forward-pass a batch of perturbed images in small chunks"""
        outputs = []
        n = numpy_images.shape[0]
        for start in range(0, n, self.predict_chunk_size):
            chunk = numpy_images[start:start + self.predict_chunk_size]
            tensor = torch.from_numpy(chunk).float().to(self.device)
            tensor = tensor / 255.0
            tensor = tensor.permute(0, 3, 1, 2)
            tensor = self._norm_transform(tensor)
            with torch.no_grad():
                logits = self.model(tensor)
            outputs.append(logits.cpu().numpy())
            del tensor, logits, chunk
        return np.concatenate(outputs, axis=0)

    def _lime_predict_fn(self, numpy_images):
        return self._predict_in_chunks(numpy_images)

    def compute_lime(self, raw_img_uint8, target_class):
        self.lime_explainer.random_state.seed(SEED)
        explanation = self.lime_explainer.explain_instance(
            raw_img_uint8.astype('double'),
            self._lime_predict_fn,
            top_labels=5,
            hide_color=0,
            num_samples=self.lime_samples,
            random_seed=SEED,
        )

        if target_class not in explanation.local_exp:
            raise ValueError(
                f"LIME did not explain class {target_class} "
                f"(only explained {list(explanation.local_exp.keys())}); increase top_labels."
            )
        return explanation

    def close_lime_explanation(self, explanation):
        """Explicitly drop LIME's internal segment/perturbation state."""
        explanation.segments = None
        explanation.local_exp = None
        explanation.local_pred = None

    def get_lime_heatmap(self, explanation, target_class):
        dict_heatmap = dict(explanation.local_exp[target_class])
        heatmap = np.vectorize(dict_heatmap.get)(explanation.segments)
        heatmap = self.normalize_map(heatmap)
        return self.resize_map(heatmap)

    def _shap_predict_fn(self, numpy_images):
        return self._predict_in_chunks(numpy_images)

    def _build_shap_explainer(self, image_shape):
        np.random.seed(SEED)
        random.seed(SEED)
        torch.manual_seed(SEED)
        masker_blur = shap.maskers.Image("blur(32,32)", image_shape)
        self.shap_explainer = shap.Explainer(self._shap_predict_fn, masker_blur, output_names=self.class_names)

    def compute_shap(self, raw_img_uint8, target_class=None, all_classes=True):
        if self.shap_explainer is None:
            self._build_shap_explainer(raw_img_uint8.shape)

        images_batch = raw_img_uint8[np.newaxis, ...]
        n_classes = len(self.class_names) if self.class_names else 5
        outputs = list(range(n_classes)) if all_classes else [target_class]

        return self.shap_explainer(images_batch, max_evals=self.shap_max_evals, batch_size=self.shap_batch_size, outputs=outputs)

    def get_shap_heatmap(self, shap_values, class_position=0):
        raw_shap = shap_values.values[0, :, :, :, class_position]
        sv_signed = raw_shap.sum(axis=-1)
        heatmap = self.normalize_map(sv_signed)
        return self.resize_map(heatmap), self.resize_map(sv_signed)


In [12]:
_engine_cache = {}

def get_engine_for(model):
    """Look up the AttributionEngine that was built for this exact model object."""
    eng = _engine_cache.get(id(model))
    if eng is None:
        raise RuntimeError("No AttributionEngine registered for this model.")
    return eng


In [13]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def get_predicted_labels(model, samples, device, transform=test_transform):
    preds = []
    model.eval()
    with torch.no_grad():
        for raw_img_np, _ in samples:
            input_tensor = transform(PILImage.fromarray(raw_img_np)).unsqueeze(0).to(device)
            pred = model(input_tensor).argmax(dim=1).item()
            preds.append(pred)
    return np.array(preds)


**Max-sensitivity xai functions**

In [14]:
def explain_gradcam(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        raw = eng.unnormalize_to_uint8(torch.from_numpy(img).float())  # same clip + truncation as LIME/SHAP
        x = eng.uint8_to_model_input(raw)
        maps.append(eng.compute_gradcam(x, int(target)))
    return np.stack(maps)[:, None, :, :]

In [15]:
def explain_gradcam(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        raw = eng.unnormalize_to_uint8(torch.from_numpy(img).float())
        x=eng.uint8_to_model_input(raw)
        gradcam = eng.compute_gradcam(x,int(target))
        maps.append(gradcam)
    return np.stack(maps)[:, None, :, :]


def explain_lime(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        explanation = eng.compute_lime(raw, int(target))
        heatmap = eng.get_lime_heatmap(explanation, int(target))
        maps.append(heatmap)
        eng.close_lime_explanation(explanation)
        del explanation
    return np.stack(maps)[:, None, :, :]

def explain_shap(model, inputs, targets, **kwargs):
    maps = []
    eng = get_engine_for(model)
    for img, target in zip(inputs, targets):
        img_tensor = torch.from_numpy(img).float()
        raw = eng.unnormalize_to_uint8(img_tensor)
        shap_values = eng.compute_shap(raw, target_class=int(target), all_classes=False)
        heatmap, _ = eng.get_shap_heatmap(shap_values, class_position=0)
        maps.append(heatmap)
    return np.stack(maps)[:, None, :, :]


# Kept as an ordered dict-of-one so we can loop "one method at a time" deliberately below
explain_funcs = {
    "GradCAM": explain_gradcam,
    # "LIME": explain_lime,
    # "SHAP": explain_shap,
}


**Checkpointing**

In [16]:
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "maxsens_per_image_checkpoint.csv")

CHECKPOINT_COLUMNS = ["model","method","image_index","max_sensitivity","ssim_dist","spearman_dist"]


def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=CHECKPOINT_COLUMNS)


def save_row(row):
    pd.DataFrame([row], columns = CHECKPOINT_COLUMNS).to_csv(
        CHECKPOINT_PATH, mode= "a", header = not os.path.exists(CHECKPOINT_PATH), index=False)

def append_checkpoint_row(row: dict):
    """Append a single completed (model, method, image_index) result to disk immediately."""
    df_row = pd.DataFrame([row], columns=CHECKPOINT_COLUMNS)
    write_header = not os.path.exists(CHECKPOINT_PATH)
    df_row.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def already_done(checkpoint_df, model_name, method, image_index):
    return ((checkpoint_df["model"] == model_name) & (checkpoint_df["method"] == method)
            & (checkpoint_df["image_index"] == image_index)).any() 


**Max-sensitivity configuration loop**

In [17]:
history = []

def custom_multi_similarity(a, b, **kwargs):
    # FROBENIUS distance ratio
    diff = quantus.similarity_func.difference(a,b)
    diff_ratio = float(norm_func.fro_norm(diff)[0]/norm_func.fro_norm(a)[0])

    # SPEARMAN RANKING
    a_flat = np.nan_to_num(a).astype(np.float32).ravel()
    b_flat = np.nan_to_num(b).astype(np.float32).ravel()
    
    rho, _ = spearmanr(a_flat, b_flat)
    spearman_dist = 1.0 - float(rho) if not np.isnan(rho) else 1.0

    a2, b2 = a.reshape(224, 224), b.reshape(224,224)
    ssim_dist = 1.0 - quantus.similarity_func.ssim(a2,b2)

    history.append((diff_ratio, ssim_dist, spearman_dist))

    return diff

In [18]:
MAXSENS_KWARGS = dict(
    nr_samples=10,
    lower_bound=0.1, #We set it below the library default as EfficeintNetB0 is a very compact model -> sensitive to pertubations in the input
    similarity_func=custom_multi_similarity,
)

MODEL_NAMES = ["resnet50", "efficientnetb0"]
# METHOD_NAMES = ["GradCAM", "LIME", "SHAP"] 
METHOD_NAMES = ["GradCAM"]

In [19]:
checkpoint_df = load_checkpoint()
print(f"Resuming from checkpoint with {len(checkpoint_df)} previously completed (model, method, image) rows.")

for model_name in MODEL_NAMES:
    print(f"\n===== {model_name} =====")

    #Loading model
    drive_weight_path = os.path.join(drive_model_dir, f"{model_name}_best_model.pth")
    model = load_model(model_name, drive_weight_path, device)

    engine = AttributionEngine(model, device,)
    _engine_cache[id(model)] = engine

    x_batch_all = np.stack([
        preprocess_image(img.astype(np.float32) / 255.0, mean=[0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]).squeeze(0).numpy()
        for img, _ in maxsens_samples
    ])

    y_batch_all = get_predicted_labels(model, maxsens_samples, device)
    gpu_mem_report(f"after loading {model_name}")

    x = torch.from_numpy(x_batch_all[0]).float()
    raw = engine.unnormalize_to_uint8(x)
    x2 = engine.uint8_to_model_input(raw)[0].cpu()
    print("===========TEST===========")
    print((x - x2).abs().max())
    print("===========TEST===========")

    # Process one xai method at a time to avoid OOM errors
    for method in METHOD_NAMES:
        print(f"\n  Method: {method} ({model_name})")

        explain_func = explain_funcs[method]
        metric = MaxSensitivity(**MAXSENS_KWARGS)
        set_seed(SEED)

        for img_idx in tqdm(range(len(x_batch_all)), desc = f"{model_name}/{method}"):
            if already_done(checkpoint_df, model_name, method, img_idx):
                continue  
            
            # Isolating one image from the batch of images to run our test on
            x_single = x_batch_all[img_idx:img_idx + 1]
            y_single = y_batch_all[img_idx:img_idx + 1]
            row = dict(model=model_name, method=method, image_index=img_idx,
                       max_sensitivity = np.nan, ssim_dist = np.nan, spearman_dist= np.nan)

            try:
                a_single = explain_func(model, x_single, y_single)

                if np.all(a_single==0):
                    print("THERE IS A ZERO EXPLANATION!")
                else:
                    history.clear()

                    score = float(metric(model=model,
                                         x_batch = x_single,
                                         y_batch = y_single,
                                         a_batch = a_single,
                                         device = device,
                                         explain_func=explain_func)[0])    
                    row["max_sensitivity"] = score

                    s = np.array(history, dtype=float)
                    if len(s) and not np.isnan(score):
                        k = np.argmin(np.abs(s[:,0]-score))
                        row["ssim_dist"], row["spearman_dist"] = s[k,1], s[k,2]

            except AssertionError as e:
                print(f"{model_name} had a degenerate explanation on image {img_idx} for {method}: {e}")
            
            except Exception as e:
                print(f"    Error on {model_name}/{method}/image {img_idx}: {e}")
                continue

            save_row(row)
            checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame([row])], ignore_index = True)
            if img_idx % 10 ==0:
                clear_memory()

        del metric
        clear_memory()

    del x_batch_all, y_batch_all

    # Clear the model completely before loading the next one
    unload_model(model=model, engine=engine, engine_cache=_engine_cache)
    gpu_mem_report(f"after unloading {model_name}")


Resuming from checkpoint with 0 previously completed (model, method, image) rows.

===== resnet50 =====


[mem after loading resnet50] allocated=0.10 GB  reserved=0.16 GB
===========TEST===========


tensor(0.0175)
===========TEST===========

  Method: GradCAM (resnet50)


[mem after unloading resnet50] allocated=0.02 GB  reserved=0.04 GB

===== efficientnetb0 =====


[mem after loading efficientnetb0] allocated=0.03 GB  reserved=0.06 GB
===========TEST===========
tensor(0.0175)
===========TEST===========

  Method: GradCAM (efficientnetb0)


efficientnetb0 had a degenerate explanation on image 54 for GradCAM: The elements in the attribution vector are all equal to zero, which may cause inconsistent results since many metrics rely on ordering. Recompute the explanations.


[mem after unloading efficientnetb0] allocated=0.02 GB  reserved=0.04 GB


**Summary and plot results**

In [20]:
full_maxsens_df = load_checkpoint()
summary = (full_maxsens_df.groupby(["model", "method"])
             .agg(mean_maxsens=("max_sensitivity", "mean"),
                  std_maxsens=("max_sensitivity", "std"),
                  mean_ssim_dist=("ssim_dist", "mean"),
                  mean_spearman_dist=("spearman_dist", "mean"),
                  n_valid=("max_sensitivity", "count"),
                  n_failed=("max_sensitivity", lambda s: s.isna().sum()))
             .round(4).reset_index())
display(summary)
summary.to_csv(os.path.join(RESULTS_DIR, f"maxsens_summary.csv"), index=False)

,model,method,mean_maxsens,std_maxsens,mean_ssim_dist,mean_spearman_dist,n_valid,n_failed
0,efficientnetb0,GradCAM,1.0284,0.2498,0.7347,0.9090,80,1
1,resnet50,GradCAM,0.8591,0.2527,0.5854,0.6263,81,0


In [21]:
# plt.figure(figsize=(9, 5))
# pivot = maxsens_summary.pivot(index='method', columns='model', values='mean_max_sensitivity')
# pivot.plot(kind='bar', ax=plt.gca())
# plt.yscale("log")
# plt.ylabel("Mean MaxSensitivity (log scale, lower = more robust)")
# plt.title("MaxSensitivity Comparison by Model and Method")
# plt.xticks(rotation=0)
# plt.grid(axis="y", alpha=0.3)
# plt.tight_layout()
# plt.show()
